# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.3 MB/s eta 0:00:00:00:0100:01


In [2]:
from datasets import load_dataset
import pandas as pd
import re       # libreria de expresiones regulares
import string   # libreria de cadena de caracteres
from gensim.models.phrases import Phrases, Phraser
import multiprocessing
from gensim.models import Word2Vec
import numpy as np

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
sinopsis = load_dataset("mathigatti/spanish_imdb_synopsis")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

plots.csv:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4967 [00:00<?, ? examples/s]

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [64]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/usuarios/usuarios.csv")

In [65]:
usuarios

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...
5,U06,Martín,definido,Bienvenidos a Collinwood,El gran golpe,L.A. Confidential,Sympathy for Mr. Vengeance,La otra cara del crimen,Un grupo de personas planea un robo o estafa y...
6,U07,Sofía,definido,Velvet Goldmine,"Cuanto más, ¡mejor!",La vida de bohemia,Cero en conducta,Corazón salvaje,Una película sobre músicos o artistas que vive...
7,U08,Diego,definido,Superdetective en Hollywood,Mission: Impossible,Misión: Imposible 3,"Walker, Texas Ranger",300,Acción directa con un héroe que trabaja solo o...
8,U09,Elena,definido,Viaje a Darjeeling,Mi Idaho privado,Melinda y Melinda,La ciencia del sueño,Un beso,Algo tranquilo sobre personas que intentan rec...
9,U10,Facundo,definido,Sátántangó,Corazón salvaje,Mi Idaho privado,La ciencia del sueño,Sympathy for Mr. Vengeance,"Algo que sea difícil de clasificar, con una ló..."


Visualizamos las queries

In [66]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [67]:
df_pelis = pd.DataFrame(sinopsis['train'])
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

Defino una funcion para limpiar texto

In [68]:
def limpiar_texto(text):
    # pasa las mayusculas del texto a minusculas
    text = text.lower()
    # reemplaza texto entre corchetes por espacio en blanco
    text = re.sub(r'\[.*?¿\]%', ' ', text)
    # reemplaza signos de puntuacion por espacio en blanco
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # remueve palabras que contienen numeros.
    text = re.sub(r'\w*\d\w*', '', text)
    # remueve caracteres especiales y saltos de linea
    text = re.sub('[‘’“”…«»]', '', text)
    text = re.sub('\n', ' ', text)
    return text

Unificamos las variables relevantes en un texto

In [69]:
df_pelis["texto"] = (
    df_pelis["name"] + " "
    + df_pelis["description"] + " "
    + df_pelis["year"].fillna('').astype(str) + " "
    + df_pelis["director"].fillna('') + " "
    + df_pelis["genre"] + " "
    + df_pelis["keywords"]
)

In [70]:
limpieza = lambda x: limpiar_texto(x)
data_clean = pd.DataFrame(df_pelis["texto"].apply(limpieza))

Agregamos bigramas al corpus (?)

In [71]:
input = [row.split() for row in data_clean["texto"]] # separamos en una lista
phrases = Phrases(input, min_count=20, progress_per=1000)

bigram = Phraser(phrases)

sentences = bigram[input]

## Embedding de peliculas

### Entrenamos modelo World2Vec

> [!!!] falta elegir los parametros del modelo acorde al trabajo.

In [72]:
cores = multiprocessing.cpu_count()

w2v_model = Word2Vec(min_count=20, # ignora palabras cuya frecuencia es menor a esta
                     window=2, # tamanio de la ventana de contexto
                     vector_size=300, # dimension del embedding
                     sample=6e-5, # umbral para downsamplear palabras muy frecuentes
                     alpha=0.03, # tasa de aprendizaje inicial (entrenamiento de la red neuronal)
                     min_alpha=0.0007, # tasa de aprendizaje minima
                     negative=20, # penalidad de palabras muy frecuentes o poco informaitvas
                     workers=cores) # numero de cores para entrenar el modelo

w2v_model.build_vocab(sentences, progress_per=10000) # construye el vocabulario

### ENTRENA EL MODELO
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

(1273223, 6144450)

### Calcular vector promedio de cada película

Definimos una funcion que calcula el vector promedio a partir de un texto

In [73]:
def obtener_vector_promedio(texto, modelo):
    palabras = texto.split()

    vectores_palabras = [modelo.wv[palabra] for palabra in palabras if palabra in modelo.wv]

    if not vectores_palabras:
        return np.zeros(modelo.wv.vector_size)

    return np.mean(vectores_palabras, axis=0)

Calculamos el embedding promedio de cada pelicula

In [74]:
embeddings_peliculas = pd.DataFrame(np.array([obtener_vector_promedio(text, w2v_model) for text in data_clean['texto']]))

In [75]:
embeddings_peliculas.head()

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.026860,-0.043123,0.109532,0.160626,-0.090130,0.041444,-0.083698,0.283033,0.046298,-0.003334,...,0.022414,0.078168,0.353141,-0.106382,0.184533,0.058586,0.128227,-0.053880,0.130868,0.046801
1,0.053745,-0.032911,0.125478,0.265222,-0.121534,0.041227,-0.126677,0.286623,-0.007996,-0.000666,...,0.078355,0.145782,0.286818,-0.092366,0.188506,0.082057,0.123319,-0.050438,0.123135,0.040146
2,0.036675,-0.038163,0.060766,0.257068,-0.123629,0.020469,-0.130891,0.276727,0.002645,-0.044469,...,0.068150,0.140865,0.260561,-0.078657,0.209624,0.076519,0.084624,-0.041954,0.098470,0.007041
3,0.048959,-0.020380,0.098659,0.253146,-0.109256,0.024157,-0.123365,0.306333,0.008375,-0.029085,...,0.070992,0.137177,0.275823,-0.092202,0.187238,0.086825,0.101264,-0.055871,0.110311,0.039091
4,0.048403,-0.055145,0.137689,0.262213,-0.116330,0.042662,-0.139813,0.259112,0.007390,0.009581,...,0.073878,0.139359,0.313775,-0.092350,0.193923,0.066882,0.124466,-0.035700,0.125390,0.042075


Estan en orden entonces podemos joinear por index

In [76]:
pelis_embd = embeddings_peliculas.merge(df_pelis[["name","id"]], left_index=True, right_index=True)
pelis_embd.head()

,0,1,2,3,4,5,6,7,8,9,...,292,293,294,295,296,297,298,299,name,id
0,0.026860,-0.043123,0.109532,0.160626,-0.090130,0.041444,-0.083698,0.283033,0.046298,-0.003334,...,0.353141,-0.106382,0.184533,0.058586,0.128227,-0.053880,0.130868,0.046801,Herida abierta,1
1,0.053745,-0.032911,0.125478,0.265222,-0.121534,0.041227,-0.126677,0.286623,-0.007996,-0.000666,...,0.286818,-0.092366,0.188506,0.082057,0.123319,-0.050438,0.123135,0.040146,"Elvira, reina de las tinieblas",2
2,0.036675,-0.038163,0.060766,0.257068,-0.123629,0.020469,-0.130891,0.276727,0.002645,-0.044469,...,0.260561,-0.078657,0.209624,0.076519,0.084624,-0.041954,0.098470,0.007041,Durmiendo con su enemigo,3
3,0.048959,-0.020380,0.098659,0.253146,-0.109256,0.024157,-0.123365,0.306333,0.008375,-0.029085,...,0.275823,-0.092202,0.187238,0.086825,0.101264,-0.055871,0.110311,0.039091,Elizabethtown,4
4,0.048403,-0.055145,0.137689,0.262213,-0.116330,0.042662,-0.139813,0.259112,0.007390,0.009581,...,0.313775,-0.092350,0.193923,0.066882,0.124466,-0.035700,0.125390,0.042075,Godzilla,5


## Embeddings de usuarios

Limpiamos las queries con el mismo proceso de antes

In [77]:
data_clean_users = pd.DataFrame(usuarios["query"].apply(limpieza))

### Embedding de las queries

In [78]:
embeddings_query = pd.DataFrame()

for query in data_clean_users["query"]:
    words = query.split()
    words_embeddings = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    embedding_mean = np.mean(words_embeddings, axis=0)
    embeddings_query = pd.concat([embeddings_query, pd.DataFrame(embedding_mean).T])

embeddings_query.reset_index(drop=True,inplace=True)

In [79]:
embeddings_query = embeddings_query.merge(usuarios[["id"]], left_index=True, right_index=True)
embeddings_query

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,id
0,0.034363,-0.043872,0.092217,0.301811,-0.149023,0.018383,-0.117616,0.296884,-0.026853,-0.030810,...,0.162302,0.274544,-0.108077,0.192799,0.085022,0.121814,-0.039916,0.120147,0.027404,U01
1,0.025925,-0.045655,0.114962,0.256702,-0.108161,0.045830,-0.120653,0.278886,-0.001601,-0.014695,...,0.145953,0.272653,-0.100671,0.196849,0.085920,0.107219,-0.057091,0.114883,0.033319,U02
2,0.037390,-0.053837,0.099893,0.304289,-0.137858,0.024172,-0.134384,0.306110,-0.026609,-0.021517,...,0.164354,0.267267,-0.111067,0.215865,0.084152,0.119494,-0.050047,0.124098,0.025444,U03
3,0.054688,-0.066090,0.120887,0.310190,-0.145087,0.039859,-0.141021,0.290710,-0.026201,-0.029872,...,0.197068,0.278863,-0.103546,0.211196,0.085370,0.113718,-0.029133,0.157812,0.029124,U04
4,0.058453,-0.033818,0.151597,0.270934,-0.123217,0.044281,-0.133031,0.292621,-0.004083,0.000218,...,0.164299,0.290506,-0.104895,0.187640,0.089996,0.125156,-0.043009,0.151311,0.044965,U05
5,0.024181,-0.077253,0.118473,0.276461,-0.145928,0.045304,-0.133809,0.298112,-0.013517,-0.034529,...,0.183055,0.295984,-0.114268,0.203512,0.064071,0.131998,-0.033645,0.148513,0.033361,U06
6,0.045575,-0.044468,0.109107,0.315848,-0.146266,0.025533,-0.138240,0.302646,-0.031201,-0.011132,...,0.179116,0.260373,-0.101915,0.210857,0.092673,0.115697,-0.039670,0.127274,0.020071,U07
7,0.046633,-0.066067,0.138375,0.260394,-0.129292,0.046749,-0.132376,0.280696,0.003727,-0.017851,...,0.163329,0.317417,-0.106038,0.192041,0.058860,0.122601,-0.034125,0.154311,0.042709,U08
8,0.029313,-0.085742,0.119625,0.323646,-0.158700,0.040394,-0.136571,0.303347,-0.035626,-0.030035,...,0.201649,0.283578,-0.124903,0.215953,0.080098,0.135408,-0.033531,0.152402,0.030049,U09
9,0.044226,-0.063081,0.098895,0.317615,-0.155791,0.024045,-0.127747,0.308939,-0.035131,-0.038188,...,0.195756,0.263841,-0.111178,0.210841,0.089824,0.117494,-0.030085,0.148907,0.027224,U10


### Embedding historial

In [ ]:
def calcular_embedding_historial(usuario, pelis_embd):
    peliculas_usuario = usuario[['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']].tolist()
    embeddings = []
    for pelicula in peliculas_usuario:
        # check pelicula in pelis_embd
        if pelicula not in pelis_embd['name'].values:
            print(f"Película '{pelicula}' no encontrada en el DataFrame de embeddings.")
            continue # se omite del calculo del embedding del historial
        embedding = pelis_embd[pelis_embd['name'] == pelicula].iloc[0][:-2].to_numpy(dtype=np.float32)
        embeddings.append(embedding)
    historial_embedding = np.mean(embeddings, axis=0)
    return historial_embedding

In [81]:
historiales_embeddings = []
for index, usuario in usuarios.iterrows():
    historial_embedding = calcular_embedding_historial(usuario, pelis_embd)
    historial_embedding = np.append(historial_embedding, usuario['id'])
    
    historiales_embeddings.append(historial_embedding)

historiales_df = pd.DataFrame(historiales_embeddings)

Película 'Rec' no encontrada en el DataFrame de embeddings.
Película 'El secreto de sus ojos' no encontrada en el DataFrame de embeddings.
Película 'El exorcista' no encontrada en el DataFrame de embeddings.
Película 'Intocable' no encontrada en el DataFrame de embeddings.
Película 'Una mente brillante' no encontrada en el DataFrame de embeddings.
Película 'Kill Bill: Volume 1' no encontrada en el DataFrame de embeddings.
Película 'Paddington' no encontrada en el DataFrame de embeddings.


Podemos agregarlas ya que no tiene sentido que falten

In [ ]:
embeddings_query_jose = embeddings_query[embeddings_query['id'] == jose['id']]
jose_promedio = np.mean([embeddings_query_jose.iloc[0][:-1].to_numpy(dtype=np.float32), historial_jose], axis=0)
jose_promedio

array([ 2.07595062e-02,  1.69602394e-01,  4.33057249e-02, -1.23935968e-01,
        1.57278582e-01, -8.35606083e-02,  7.35978782e-02,  1.16239160e-01,
        4.73516881e-02,  5.12003480e-03, -1.24026053e-02, -8.79660025e-02,
        4.77575883e-02,  1.08049653e-01, -1.57699585e-01, -1.32207215e-01,
        2.14662567e-01, -3.54450271e-02,  3.53505686e-02,  4.55073640e-02,
       -3.00550200e-02,  2.17229594e-04,  1.84994310e-01, -4.74146195e-02,
        1.75365746e-01, -1.02013245e-01, -1.08271256e-01,  1.25407711e-01,
       -9.81374923e-03, -9.54016112e-03,  1.42388195e-01, -1.87269244e-02,
       -1.02935284e-01,  1.25134036e-01, -2.22097412e-02, -1.27624767e-02,
        1.95628345e-01, -3.28973651e-01, -3.16463597e-02, -1.88383043e-01,
       -1.22352399e-01, -3.52056623e-02,  4.11990210e-02,  5.05440235e-02,
        7.77541101e-02,  5.24775535e-02, -4.94123474e-02,  1.92725006e-03,
       -1.02063604e-01,  1.85610324e-01,  8.43111277e-02, -1.09585784e-02,
       -1.58922702e-01,  

In [ ]:
similares = w2v_model.wv.most_similar(positive=[jose_promedio],topn=10)
similares

[('varias', 0.9908871054649353),
 ('estilo', 0.9905944466590881),
 ('infancia', 0.9897976517677307),
 ('salir', 0.9887130260467529),
 ('bella', 0.9886916279792786),
 ('corazón', 0.9886060953140259),
 ('extraña', 0.9885649085044861),
 ('ir', 0.9885553121566772),
 ('seis', 0.9882989525794983),
 ('habitación', 0.9882380366325378)]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Extract movie embeddings from pelis_embd (assuming the last two columns are 'name' and 'id')
movie_embeddings = pelis_embd.iloc[:, :-2].to_numpy(dtype=np.float32)

# Calculate cosine similarity between jose_promedio and all movie embeddings
# Reshape jose_promedio to be 2D for cosine_similarity function
similarities = cosine_similarity(jose_promedio.reshape(1, -1), movie_embeddings)

# Get the indices of the top 10 most similar movies (descending order)
top_10_indices = similarities.argsort()[0][-10:][::-1]

# Get the corresponding movie names and their similarity scores
similar_movies_df = pd.DataFrame({
    'movie_name': pelis_embd.loc[top_10_indices, 'name'].values,
    'similarity_score': similarities[0, top_10_indices]
})

print("Top 10 Most Similar Movies for Jose:")
display(similar_movies_df)

Top 10 Most Similar Movies for Jose:


,movie_name,similarity_score
0,Más fuerte que su destino,0.998254
1,Todas contra él,0.998119
2,La angustia del miedo,0.998010
3,Un ángel en mi mesa,0.997948
4,El efecto mariposa,0.997947
5,Otoño en Nueva York,0.997826
6,Exorcismo en Connecticut,0.997742
7,Antes de amanecer,0.997740
8,Cuando cae la noche,0.997717
9,Conociendo a Matsuko,0.997674


## Opcion 1:
 Promedio del query con promedio de pelicula del historial contra promedio de pelicula.

## Opcion 2:
  Ponderar Promedio de query junto con el historial de pelicula, y compararlo con el promedio de pelicula.

## Opcion 3:
  Ponerle un peso al historial de peliculas por orden de visualizacion y compararlo con el promedio de pelicula.